# I1 — 판정기 입력 개선: 근거 예산 채우기 + 앞 문장

## 왜 하나

F1 base(배포 판정기)는 근거를 **40단어 조각 중 비슷한 8개**만 보고, 판정 텍스트에 **앞 문장이 없다**(SSOT §5 F1 한계).
8개 × 40단어 ≈ 420토큰에 판정 텍스트를 더하면 512토큰에 가까워, 넘치면 **유사도와 무관하게 원문 뒤쪽 조각이 잘린다**.
조사(11/04)에서 RAGTruth로 학습하지 않고도 높은 성능을 낸 ORION(arXiv 2504.15771)은 "입력 창을 근거로 꽉 채우는 것"을 핵심으로 꼽았다.
재학습 1회·지연 변화 없이(입력 길이 512 그대로) 서비스 탐지율을 올릴 수 있는지 확인한다. 10/28 결정의 "서비스 성능 향상" 후보 (2)에 해당한다.

## 무엇을 하나 (F1 base와 입력만 다름)

| | F1 base | I1 |
|---|---|---|
| 근거 | 비슷한 조각 top-8, 원문 순서, 넘치면 뒤쪽 잘림 | 비슷한 조각부터 **512토큰 예산이 찰 때까지**, 원문 순서 (잘리지 않음) |
| 판정 텍스트 | 문장 앞부분 | **앞 문장(끝 48단어) [SEP] 문장 앞부분** (응답 첫 문장은 앞부분만) |
| 검색 질의 | 문장 앞부분 | 같음 |
| 백본·학습 정답·학습 루프·시드·체크포인트 선택 | DeBERTa-v3-base, R4b 보정 정답, R1c 루프, 42, cal-sub | 같음 |

## 사전 정의 기준 (결과 보기 전, SSOT §6 I1에 고정)

비교는 **cal-L에서만**(F1 base 점수 = `final/scores_base.npy`, 재실행 배포 가중치). 임계값은 각 모델의 cal-L 등각 보정 서비스 10%(멀쩡한 응답 표시 ≤ 10%).
- **I1-0 입력 점검**: I1 입력 중 512토큰을 넘어 잘리는 비율 ≤ 1% (예산 채우기 구현 확인. 미달이면 버그로 보고 결과 해석 보류)
- **I1-1 (주)**: 8토큰 재판정, 원래 정답에서 탐지율 차이(I1 − F1 base) **≥ +0.03, 그리고** 응답 단위 쌍체 부트스트랩 95% 하한 > 0
- **I1-2 (안전)**: 보정 정답(C1)에서 탐지율 차이 ≥ −0.01, 그리고 원래 정답 탐지 지연 중앙값 증가 ≤ 2토큰
- **I1-3 (지연)**: GPU 판정 1회(검색·예산 채우기 포함) p95 ≤ 68ms (8토큰 × 8.5ms)
- **채택** = I1-1·I1-2·I1-3 모두 충족. 채택하면 게이트웨이 판정기 입력을 바꾼다(SPEC → 테스트 → 코드, S4). dev-L은 **채택 판정 후 보고만** 한다.
- 단서: 시드 1개. F1 재실행에서 같은 시드 재학습 간 변동이 약 0.015~0.018이어서 주 기준을 그 2배인 +0.03으로 둔다. cal-sub(cal-L 300응답)는 두 모델 모두 체크포인트 선택에 썼다.

> **런타임: A100.** 학습 약 90분 + 입력 구성·채점 약 20분. 드라이브에 F1과 같은 입력(`r0/`, `r4/`, `r4b/`)과 `final/scores_base.npy`가 있어야 한다.
> 결과는 드라이브 `Colab Notebooks/HalluGate/i1/`과 `i1_judge.zip`(자동 다운로드).


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/HalluGate' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = I1_DIR = os.path.join(OUT_DIR, 'i1')
FINAL_DIR = os.path.join(OUT_DIR, 'final')
R1B_DIR, R1C_DIR, R4_DIR, R4B_DIR = (os.path.join(OUT_DIR, d) for d in ['r1b', 'r1c', 'r4', 'r4b'])
os.makedirs(R1_DIR, exist_ok=True)
for p in [os.path.join(R4B_DIR, 'labels_train.json'), os.path.join(R4_DIR, 'r4_corrected_labels.json')]:
    assert os.path.exists(p), f'{p} 없음'
assert os.path.exists(os.path.join(FINAL_DIR, 'scores_base.npy')), 'final/scores_base.npy 없음 (F1 재실행 점수, 비교 기준)'
import shutil
_t = os.path.join(I1_DIR, '_write_test.bin')
try:
    with open(_t, 'wb') as f: f.write(os.urandom(50 * 2**20))
    ok = os.path.getsize(_t) == 50 * 2**20; os.remove(_t)
except OSError as e:
    ok = False; print('드라이브 쓰기 실패:', e)
print('드라이브 50MB 쓰기', '정상' if ok else '실패 — 휴지통 비우기·용량 확인. 가중치는 마지막 셀 zip으로 받는다')
print('device', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

## 2. 데이터 (F1과 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

## 3. 보정 학습 정답 (F1과 같음)

In [ ]:
TRAIN_ITEMS = to_items(train)
ITEMS = {f'tr-{x["id"]}': x for x in TRAIN_ITEMS}
LABELS = {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(os.path.join(R4B_DIR, 'labels_train.json'))).items()}
print(f'학습 재판정 {len(LABELS):,}/{len(ITEMS):,}')

In [ ]:
def content_starts(x):
    t = x['text']
    cs = [a + (len(t[a:b]) - len(t[a:b].lstrip())) for a, b in zip(x['ts'], x['te'])]
    return [c if c < b else a for c, a, b in zip(cs, x['ts'], x['te'])]

def locate(x, sn, phrase, cs):
    seg = x['text'][sn['a']:sn['b']]
    ph = (phrase or '').strip()
    k = seg.find(ph) if ph else -1
    if k < 0 and ph: k = seg.lower().find(ph.lower())
    if k < 0: return sn['t0']
    a, b = sn['a'] + k, sn['a'] + k + len(ph)
    for t in range(sn['t0'], sn['t1'] + 1):
        if min(x['te'][t], b) > max(cs[t], a): return t
    return sn['t0']

ADD_ONSET, flip, earlier = {}, 0, 0
for cid, x in ITEMS.items():
    labs = LABELS.get(cid)
    if not labs: continue
    cs, add = None, []
    for i, sn in enumerate(x['sents']):
        if sn['y'] == 0 and labs[i + 1]['label'] == 'H':
            cs = cs or content_starts(x)
            t = locate(x, sn, labs[i + 1]['phrase'], cs); ADD_ONSET[(x['id'], i)] = t; add.append(t)
    if add:
        if x['onset'] is None: flip += 1
        elif min(add) < x['onset']: earlier += 1
n_clean = sum(x['onset'] is None for x in TRAIN_ITEMS); n_hall = len(TRAIN_ITEMS) - n_clean
print(f'추가된 환각 문장 {len(ADD_ONSET):,} | 멀쩡 → 환각 {flip}/{n_clean} ({flip / max(n_clean, 1):.0%}) | 더 이른 환각 {earlier}/{n_hall} ({earlier / max(n_hall, 1):.0%})')

## 4. 학습 판정 지점 (F1과 같음)

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


WIN_TOK = 10
train_items = TRAIN_ITEMS
by_tr = {str(r['id']): r for r in train}
TR, P_KEY, P_TXT, P_CAT = [], [], [], []
for x in train_items:
    text = x['text']
    spans = [(l['start'], l['end']) for l in (by_tr[x['id']].get('labels') or [])]
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a for a, b in zip(x['ts'], x['te'])]
    def s_onset(sn):                      # E2와 같다: 이 문장 안에서 환각 구간과 처음 겹치는 토큰
        for t in range(sn['t0'], sn['t1'] + 1):
            if any(min(x['te'][t], b) > max(cs[t], a) for a, b in spans): return t
        return None
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    son = [s_onset(sn) for sn in x['sents']]
    son = [o if o is not None else ADD_ONSET.get((x['id'], i)) for i, o in enumerate(son)]   # R4b 보정 학습 정답
    groups = {}
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        if o is None: cat = 'neg'
        elif t < o: cat = 'pre'
        elif t <= o + WIN_TOK: cat = 'win'
        else: cat = 'post'
        g = len(P_TXT); P_KEY.append(x['ref_key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win:                       # 창 안에 지점이 없으면 시작 이후 첫 지점을 창으로
            P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
print(f'학습 응답 {len(TR)} | 판정 지점 {len(P_TXT):,} | 환각 문장(다중 사례 단위) {sum(len(r["groups"]) for r in TR):,} |',
      pd.Series(P_CAT).value_counts().to_dict())
assert sum(len(r['groups']) for r in TR) > 0

## 5. 근거 조각·평가 세트 (F1과 같음, top-8 검색은 생략)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


# R0 평가 세트(cal-L, dev-L)와 기존 판정기 점수
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
Q_KEYS, Q_TEXTS = R0P['q_keys'], R0P['q_texts']
for k in set(Q_KEYS):
    REFS.setdefault(k, build_reference(sources[k]))

@torch.no_grad()
def topk_idx(keys, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            out.append(sorted(torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()))
    return out
ctx_of = lambda key, idx: ' ... '.join(get_chunks(key)[0][j] for j in idx)

t0 = time.perf_counter()
P_IDX = None   # I1: 아래 예산 채우기로 대체
print(f'학습 지점 근거 검색 {time.perf_counter()-t0:.0f}초')

# 평가 질의: cal-L·dev-L 지점만 (R0와 같은 질의 번호)
EVAL_Q = sorted({q for x in CAL + DEV for _, q in x['pts2']})
t0 = time.perf_counter()
EQ_IDX = None  # I1: 아래 예산 채우기로 대체
print(f'평가 질의 {len(EVAL_Q):,}개 근거 검색 {time.perf_counter()-t0:.0f}초')

# 체크포인트 선택용 cal-sub (cal-L에서 300응답 고정)
CALSUB = random.Random(1).sample(CAL, min(300, len(CAL)))

In [ ]:
# R4 보정 정답 C1 (Opus 재판정)을 cal-L·dev-L에 적용한 사본
CORR = json.load(open(os.path.join(R4_DIR, 'r4_corrected_labels.json')))['C1']
def apply_corr(items, split):
    out = []
    for x in items:
        c = CORR[split].get(x['id'])
        y = copy.copy(x); y['sents'] = [dict(sn) for sn in x['sents']]
        if c is not None:
            y['onset'] = c['onset']
            for i in c['added']: y['sents'][i - 1]['y'] = 1
        out.append(y)
    return out
CAL_C, DEV_C = apply_corr(CAL, 'calL'), apply_corr(DEV, 'devL')
LAB_SETS = {'orig': (CAL, DEV), 'C1': (CAL_C, DEV_C)}
for k, (c, d) in LAB_SETS.items():
    print(f"{k}: cal-L 환각 {sum(x['onset'] is not None for x in c)}, dev-L 환각 {sum(x['onset'] is not None for x in d)}")
# 체크포인트 선택은 보정 정답 기준 cal-sub (같은 300응답)
_byid = {x['id']: x for x in CAL_C}
CALSUB = [_byid[x['id']] for x in CALSUB]

## 6. 평가 함수 (F1과 같음)

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

## 7. 학습 루프 (F1과 같음)

In [ ]:
KEEP_WEIGHTS = {42}
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F, shutil

BACKBONES = {'mini': ('cross-encoder/nli-MiniLM2-L6-H768', 2e-5),
             'base': ('cross-encoder/nli-deberta-v3-base', 2e-5),
             'large': ('lytang/MiniCheck-DeBERTa-v3-Large', 1e-5)}
BATCH_SEQ, MICRO, N_EVALS = 64, 16, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
if not BF16: print('⚠️ bf16 미지원 GPU — DeBERTa는 fp16에서 불안정할 수 있음 (A100/L4 권장)')

def enc_ids(tok, idxs):
    return tok([ctx_of(P_KEY[g], P_IDX[g]) for g in idxs], [P_TXT[g] for g in idxs], truncation='only_first',
               max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)

@torch.no_grad()
def score_queries(tok, model, qs, bs=128):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        enc = tok([ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b], [Q_TEXTS[q] for q in b], truncation='only_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits[:, 0].float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

Y_ALL = np.isin(P_CAT, ['win', 'post']).astype(np.float32)          # V0b: 문장 단위 시작 이후 = 1

def load_init(name):
    path, _ = BACKBONES[name]
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=1, ignore_mismatched_sizes=True).to(DEVICE)
    return tok, model

def train_backbone(name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok, model = load_init(name)
    lr = BACKBONES[name][1]
    order = list(range(len(P_TXT))); random.Random(seed).shuffle(order)
    n_steps = max(1, len(order) // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{name} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | {n_steps}단계, 학습률 {lr}')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); t0 = time.perf_counter()
    for step in range(1, n_steps + 1):
        batch = order[(step - 1) * BATCH_SEQ: step * BATCH_SEQ]
        opt.zero_grad(); tot = 0.0
        for k in range(0, len(batch), MICRO):
            mb = batch[k:k + MICRO]
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
                z = model(**enc_ids(tok, mb)).logits[:, 0].float()
            loss = F.binary_cross_entropy_with_logits(z, torch.tensor(Y_ALL[mb], device=DEVICE), reduction='sum') / len(batch)
            scaler.scale(loss).backward(); tot += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if step in eval_at:
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{name} s{seed}] 단계 {step}/{n_steps} | 손실 {tot:.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, hist

def run(name, seed):
    out = os.path.join(R1_DIR, f'{name}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{name} s{seed}] 저장이 끊긴 폴더 → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, hist = train_backbone(name, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        if seed not in KEEP_WEIGHTS:                     # 드라이브 용량: 가중치는 시드 42만 남긴다
            for f in ['model.safetensors', 'pytorch_model.bin']:
                if os.path.exists(os.path.join(out, f)): os.remove(os.path.join(out, f))
        print(f'  [{name} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{name}_s{seed}'] = np.load(sp)
    print(f'[{name} s{seed}] 준비 완료')

## 8. I1 입력 구성

학습 지점과 평가 질의마다 (앞 문장 [SEP] 문장 앞부분)과 예산 채우기 근거 조각을 만든다.
평가 질의는 앞 문장까지 포함한 새 번호를 쓰므로, cal-L·dev-L 응답의 판정 지점을 새 번호로 다시 매긴 사본을 만든다(지점·정답은 그대로).

In [ ]:
BACKBONES_PATH = BACKBONES['base'][0]

# I1 입력 구성: 앞 문장 + 근거 예산 채우기
I1_TOK = AutoTokenizer.from_pretrained(BACKBONES_PATH)
PREV_WORDS = 48          # 앞 문장은 끝 48단어까지 (긴 문장이 근거 예산을 다 먹지 않게)
SEP_COST = len(I1_TOK(' ... ', add_special_tokens=False)['input_ids'])   # 조각 사이 ' ... ' 토큰 수 (DeBERTa-v3: 3)

def text_b(prev, txt):
    if not prev:
        return txt
    return ' '.join(prev.split()[-PREV_WORDS:]) + f' {I1_TOK.sep_token} ' + txt

chunk_len_cache = {}
def chunk_lens(key):
    if key not in chunk_len_cache:
        ch, _ = get_chunks(key)
        chunk_len_cache[key] = [len(v) for v in I1_TOK(ch, add_special_tokens=False)['input_ids']]
    return chunk_len_cache[key]

def fill_select(key, e, b_len):
    # 유사도 높은 조각부터 예산(MAX_LEN − 판정 텍스트 − 특수 토큰 3개)이 찰 때까지 담고, 원문 순서로 정렬
    ch, ce = get_chunks(key)
    L = chunk_lens(key)
    budget = MAX_LEN - b_len - 3
    sel, used = [], 0
    for j in torch.argsort(ce @ e, descending=True).tolist():
        c = L[j] + (SEP_COST if sel else 0)
        if used + c <= budget:
            sel.append(j); used += c
    if not sel:                                   # 판정 텍스트가 너무 길면 가장 비슷한 조각 하나 (잘림 허용)
        sel = [int(torch.argmax(ce @ e))]
    return sorted(sel)

@torch.no_grad()
def build_inputs(keys, prevs, texts, bs=512):
    # 검색 질의는 F1과 같이 '문장 앞부분'만 쓴다 (앞 문장은 판정 입력에만)
    B = [text_b(p, t) for p, t in zip(prevs, texts)]
    blen = [len(v) for v in I1_TOK(B, add_special_tokens=False)['input_ids']]
    sels = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(list(texts[i:i + bs]), convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e, bl in zip(keys[i:i + bs], embs, blen[i:i + bs]):
            sels.append(fill_select(key, e, bl))
    return B, sels

def prev_of_train(items):
    # LABELS 셀과 같은 순서로 학습 지점마다 앞 문장을 만든다
    out = []
    for x in items:
        sidx = {sn['t0']: i for i, sn in enumerate(x['sents'])}
        for t, txt in points(x, 2):
            if not txt: continue
            si = sidx[x['sent_of'][t]]
            out.append(x['text'][x['sents'][si - 1]['a']:x['sents'][si - 1]['b']].strip() if si > 0 else '')
    return out

def build_eval_queries(items):
    # R0 질의 번호는 (근거, 텍스트)로 묶여 앞 문장이 다른 지점을 구분하지 못한다 → (근거, 앞 문장, 텍스트)로 새 번호
    index, keys, prevs, texts, MAP = {}, [], [], [], {}
    for x in items:
        d = dict(x['pts2'])
        sidx = {sn['t0']: i for i, sn in enumerate(x['sents'])}
        for t, q in x['pts2']:
            si = sidx[x['sent_of'][t]]
            pe = x['sents'][si - 1]['t1'] if si > 0 else None
            prev = Q_TEXTS[d[pe]] if pe is not None and pe in d else ''
            k = (Q_KEYS[q], prev, Q_TEXTS[q])
            if k not in index:
                index[k] = len(keys); keys.append(k[0]); prevs.append(prev); texts.append(k[2])
            MAP[(x['id'], t)] = index[k]
    return keys, prevs, texts, MAP

def remap(items, MAP):
    out = []
    for x in items:
        y = copy.copy(x); y['pts2'] = [(t, MAP[(x['id'], t)]) for t, _ in x['pts2']]
        out.append(y)
    return out

t0 = time.perf_counter()
P_PREV = prev_of_train(train_items)
assert len(P_PREV) == len(P_TXT), (len(P_PREV), len(P_TXT))
P_B, P_SEL = build_inputs(P_KEY, P_PREV, P_TXT)
print(f'학습 지점 {len(P_TXT):,}개 입력 구성 {time.perf_counter() - t0:.0f}초 | 앞 문장 있는 지점 {np.mean([bool(p) for p in P_PREV]):.1%}')

Q2_KEYS, Q2_PREV, Q2_TEXTS, Q2_MAP = build_eval_queries(CAL + DEV)
t0 = time.perf_counter()
Q2_B, Q2_SEL = build_inputs(Q2_KEYS, Q2_PREV, Q2_TEXTS)
print(f'평가 질의 {len(Q2_KEYS):,}개 (R0 번호 {len(EVAL_Q):,}개) 입력 구성 {time.perf_counter() - t0:.0f}초')
LAB_SETS_I1 = {lab: (remap(c, Q2_MAP), remap(d, Q2_MAP)) for lab, (c, d) in LAB_SETS.items()}

# I1-0 입력 점검: 잘림 비율 (I1) vs top-8 방식(F1)의 잘림 비율, 평가 질의 5,000개 표본
rs = random.Random(5).sample(range(len(Q2_KEYS)), min(5000, len(Q2_KEYS)))
n_i1 = [len(I1_TOK(ctx_of(Q2_KEYS[q], Q2_SEL[q]), Q2_B[q])['input_ids']) for q in rs]
emb = retriever.encode([Q2_TEXTS[q] for q in rs], convert_to_tensor=True, normalize_embeddings=True, batch_size=512, show_progress_bar=False)
top8 = [sorted(torch.topk(get_chunks(Q2_KEYS[q])[1] @ e, min(K, get_chunks(Q2_KEYS[q])[1].shape[0])).indices.tolist()) for q, e in zip(rs, emb)]
n_f1 = [len(I1_TOK(ctx_of(Q2_KEYS[q], ix), Q2_TEXTS[q])['input_ids']) for q, ix in zip(rs, top8)]
n_ch_i1 = [len(Q2_SEL[q]) for q in rs]
TRUNC_I1, TRUNC_F1 = float(np.mean(np.array(n_i1) > MAX_LEN)), float(np.mean(np.array(n_f1) > MAX_LEN))
I1_0 = TRUNC_I1 <= 0.01
print(f'잘림 비율: F1 방식 {TRUNC_F1:.1%} → I1 {TRUNC_I1:.1%} | I1 근거 조각 수 중앙값 {np.median(n_ch_i1):.0f} (F1 8개) | 입력 길이 중앙값 F1 {np.median(n_f1):.0f} / I1 {np.median(n_i1):.0f}')
print(f"=== I1-0 입력 점검: {'충족' if I1_0 else '미달 → 예산 채우기 버그 확인, 결과 해석 보류'} ===")

In [ ]:
# F1 학습 루프(train_backbone)는 그대로 쓰고, 입력·체크포인트 평가만 I1 입력으로 바꾼다
def enc_ids(tok, idxs):
    return tok([ctx_of(P_KEY[g], P_SEL[g]) for g in idxs], [P_B[g] for g in idxs], truncation='only_first',
               max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)

@torch.no_grad()
def score_q2(tok, model, qs, bs=128):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q2_B[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        enc = tok([ctx_of(Q2_KEYS[q], Q2_SEL[q]) for q in b], [Q2_B[q] for q in b], truncation='only_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits[:, 0].float().cpu().numpy()
    model.train()
    return out

CALSUB2 = remap(CALSUB, Q2_MAP)                       # 보정 정답(C1) cal-sub 300응답, F1과 같은 응답
CALSUB2_Q = sorted({q for x in CALSUB2 for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_q2(tok, model, CALSUB2_Q)
    SC['_cur'] = np.zeros(len(Q2_KEYS), dtype=np.float32); SC['_cur'][CALSUB2_Q] = v
    r4 = evaluate(CALSUB2, '_cur|4', calibrate(CALSUB2, '_cur|4', 100))
    r2 = evaluate(CALSUB2, '_cur|2', calibrate(CALSUB2, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

## 9. 학습·저장 (시드 42, F1 base와 같은 루프)

로컬에 fp16으로 먼저 저장하고 드라이브에 복사·크기 검증. 점수는 저장한 파일을 다시 불러 매긴다.

In [ ]:
LOCAL = '/content/i1_local'
D = os.path.join(LOCAL, 'judge_i1')
os.makedirs(LOCAL, exist_ok=True)
WFILES = ['model.safetensors', 'pytorch_model.bin']
def has_weights(d):
    return os.path.isdir(d) and any(os.path.exists(os.path.join(d, f)) and os.path.getsize(os.path.join(d, f)) > 0 for f in WFILES)
def sizes(d):
    return {f: os.path.getsize(os.path.join(d, f)) for f in sorted(os.listdir(d)) if os.path.isfile(os.path.join(d, f))}
def to_drive():
    dst = os.path.join(I1_DIR, 'judge_i1')
    try:
        shutil.copytree(D, dst, dirs_exist_ok=True)
        ok = sizes(D) == {f: v for f, v in sizes(dst).items() if f in sizes(D)}
    except OSError as e:
        print('드라이브 복사 실패:', e); ok = False
    print(f'드라이브 저장 {"확인" if ok else "실패 — 마지막 셀 zip을 꼭 내려받을 것"}')
    return ok

DD = os.path.join(I1_DIR, 'judge_i1')
if not has_weights(D) and has_weights(DD):
    shutil.copytree(DD, D, dirs_exist_ok=True); print('드라이브 저장본 사용')
if not has_weights(D):
    t0 = time.perf_counter()
    tok, model, hist = train_backbone('base', 42)
    os.makedirs(D, exist_ok=True); tok.save_pretrained(D); model.half().save_pretrained(D)
    pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(D, 'hist.csv'), index=False)
    print(f'학습 {(time.perf_counter() - t0) / 60:.0f}분')
    del model; gc.collect(); torch.cuda.empty_cache()
    DRIVE_OK = to_drive()
else:
    DRIVE_OK = has_weights(DD)

sp, spd = os.path.join(LOCAL, 'scores_i1.npy'), os.path.join(I1_DIR, 'scores_i1.npy')
if not os.path.exists(sp) and os.path.exists(spd): shutil.copy(spd, sp)
if not os.path.exists(sp):
    tok = AutoTokenizer.from_pretrained(D)
    model = AutoModelForSequenceClassification.from_pretrained(D).float().to(DEVICE)
    t0 = time.perf_counter()
    np.save(sp, score_q2(tok, model, list(range(len(Q2_KEYS)))))
    try: shutil.copy(sp, spd)
    except OSError as e: print('점수 드라이브 저장 실패', e)
    print(f'평가 질의 채점 {time.perf_counter() - t0:.0f}초')
    del model; gc.collect(); torch.cuda.empty_cache()
SC['I1'] = np.load(sp)
SC['F_base'] = np.load(os.path.join(FINAL_DIR, 'scores_base.npy'))

## 10. 채택 판정 (cal-L만)

먼저 F1 base 점수로 F1 재실행의 8토큰 서비스 임계값(3.969)이 재현되는지 확인한다(데이터·점수 불러오기 점검).

In [ ]:
ALPHA = 0.10
def conformal_flag(items, scorer, alpha=ALPHA):
    mx = np.sort([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def detect_vec(items, scorer, thr):
    # 환각 응답마다 엄격 탐지 여부(첫 경보가 발생 이후)와 지연
    det, dl = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, scorer) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        det.append(ok); dl.append(first - x['onset'] if ok else np.nan)
    return np.array(det, dtype=float), np.array(dl)

def compare(sets_b, sets_i, kind, n_boot=2000, seed=2027):
    # sets_*: {'orig': (cal, dev), 'C1': (cal, dev)} — 기준선은 R0 질의 번호, I1은 새 질의 번호로 매긴 같은 응답
    rows = {}
    for lab in ['orig', 'C1']:
        cb, ci = sets_b[lab][0], sets_i[lab][0]
        tb, ti = conformal_flag(cb, f'F_base|{kind}'), conformal_flag(ci, f'I1|{kind}')
        db, lb = detect_vec(cb, f'F_base|{kind}', tb); di, li = detect_vec(ci, f'I1|{kind}', ti)
        rng = np.random.default_rng(seed); n = len(db)
        boot = [(di[ix].mean() - db[ix].mean()) for ix in (rng.integers(0, n, n) for _ in range(n_boot))]
        rows[lab] = {'thr_base': tb, 'thr_i1': ti, 'recall_base': db.mean(), 'recall_i1': di.mean(),
                     'delta': di.mean() - db.mean(), 'ci_lo': float(np.percentile(boot, 2.5)), 'ci_hi': float(np.percentile(boot, 97.5)),
                     'delay_base': float(np.nanmedian(lb)), 'delay_i1': float(np.nanmedian(li)), 'n_hall': n}
    return rows

_chk = conformal_flag(CAL, 'F_base|8')
assert abs(_chk - 3.968751) < 1e-3, f'F1 재실행 임계값 재현 실패: {_chk}'
print(f'F1 base 8토큰 cal-L 서비스 임계값 재현: {_chk:.4f}')

CMP = {k: compare(LAB_SETS, LAB_SETS_I1, k) for k in ['end', '4', '8', '16']}
rows = [{'kind': k, 'labels': lab, **v} for k, d in CMP.items() for lab, v in d.items()]
cmp_tab = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(cmp_tab.round(3).to_string(index=False))

# I1-3 지연: 판정 1회 = 검색(임베딩) + 예산 채우기 + 교차 인코더, 배치 1 (게이트웨이 judge()와 같은 조건)
tok = AutoTokenizer.from_pretrained(D)
model = AutoModelForSequenceClassification.from_pretrained(D).float().to(DEVICE).eval()
@torch.no_grad()
def judge_once(key, prev, txt):
    e = retriever.encode([txt], convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)[0]
    b = text_b(prev, txt); bl = len(I1_TOK(b, add_special_tokens=False)['input_ids'])
    enc = tok([ctx_of(key, fill_select(key, e, bl))], [b], truncation='only_first', max_length=MAX_LEN, return_tensors='pt').to(DEVICE)
    with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
        s = float(model(**enc).logits.squeeze().float().cpu())
    if DEVICE == 'cuda': torch.cuda.synchronize()
    return s
LQ = random.Random(2027).sample(range(len(Q2_KEYS)), 200)
for q in LQ[:5]: judge_once(Q2_KEYS[q], Q2_PREV[q], Q2_TEXTS[q])
ms = []
for q in LQ:
    t0 = time.perf_counter(); judge_once(Q2_KEYS[q], Q2_PREV[q], Q2_TEXTS[q]); ms.append((time.perf_counter() - t0) * 1000)
P95 = float(np.percentile(ms, 95))
print(f'판정 1회 {torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"}: p50 {np.percentile(ms, 50):.1f} / p95 {P95:.1f} ms')
del model; gc.collect(); torch.cuda.empty_cache()

o, c1 = CMP['8']['orig'], CMP['8']['C1']
I1_1 = o['delta'] >= 0.03 and o['ci_lo'] > 0
I1_2 = c1['delta'] >= -0.01 and (o['delay_i1'] - o['delay_base']) <= 2
I1_3 = P95 <= 68
ADOPT = bool(I1_0 and I1_1 and I1_2 and I1_3)
print(f"=== I1-1 (주) 8토큰 원래 정답: {o['recall_base']:.3f} → {o['recall_i1']:.3f} (Δ {o['delta']:+.3f} [{o['ci_lo']:+.3f}, {o['ci_hi']:+.3f}]) → {'충족' if I1_1 else '미달'} ===")
print(f"=== I1-2 (안전) C1 Δ {c1['delta']:+.3f}, 지연 {o['delay_base']:.0f} → {o['delay_i1']:.0f}토큰 → {'충족' if I1_2 else '미달'} ===")
print(f"=== I1-3 (지연) p95 {P95:.1f}ms → {'충족' if I1_3 else '미달'} ===")
print(f"=== I1 채택: {'채택 → 게이트웨이 입력 변경(S4)' if ADOPT else '미채택 → F1 base 유지'} ===")

## 11. dev-L 확인 (보고만, 채택 판정에 쓰지 않음)

cal-L에서 정한 각 모델의 서비스 임계값을 dev-L에 그대로 적용한다. 과제별로도 본다.

In [ ]:
def evaluate_at(items, scorer, thr):
    r = evaluate(items, scorer, thr)
    return {'recall': r['recall'], 'delay_med': r['delay_tok_median'], 'clean_flagged': r['clean_resp_flagged']}
DEV_ROWS = []
for k in ['end', '4', '8', '16']:
    for lab in ['orig', 'C1']:
        c = CMP[k][lab]
        for model_name, sets, sc, thr in [('F_base', LAB_SETS, f'F_base|{k}', c['thr_base']), ('I1', LAB_SETS_I1, f'I1|{k}', c['thr_i1'])]:
            dev_ = sets[lab][1]
            DEV_ROWS.append({'model': model_name, 'kind': k, 'labels': lab, 'task': 'all', **evaluate_at(dev_, sc, thr)})
            if k == '8' and lab == 'orig':
                for task in sorted({x['task_type'] for x in dev_}):
                    DEV_ROWS.append({'model': model_name, 'kind': k, 'labels': lab, 'task': task,
                                     **evaluate_at([x for x in dev_ if x['task_type'] == task], sc, thr)})
dev_tab = pd.DataFrame(DEV_ROWS)
print(dev_tab.round(3).to_string(index=False))

## 12. 저장과 zip 다운로드

In [ ]:
cfg = {'name': 'judge_i1', 'backbone': BACKBONES_PATH, 'seed': 42,
       'input': {'evidence': 'similarity-ranked 40-word chunks filled up to max_len budget, original order, joined by " ... "',
                 'text_b': f'previous sentence (last {PREV_WORDS} words) [SEP] sentence prefix; first sentence: prefix only',
                 'retrieval_query': 'sentence prefix'},
       'embedder': 'sentence-transformers/all-MiniLM-L6-v2', 'max_len': MAX_LEN, 'chunk_words': CHUNK_WORDS,
       'stride_unit': 'tiktoken cl100k_base tokens, counted from sentence start (sentence end always judged)',
       'default': {'kind': '8', 'threshold': 'svc10_orig'}, 'threshold': CMP['8']['orig']['thr_i1'],
       'thresholds': {k: {f'svc10_{lab}': CMP[k][lab]['thr_i1'] for lab in ['orig', 'C1']} for k in CMP},
       'score': 'logit (higher = hallucinated)', 'weights_dtype': 'float16 (load as float32)',
       'i1': {'I1_0': bool(I1_0), 'I1_1': bool(I1_1), 'I1_2': bool(I1_2), 'I1_3': bool(I1_3), 'adopt': ADOPT}}
json.dump(cfg, open(os.path.join(D, 'judge_config.json'), 'w'), indent=1, ensure_ascii=False)
summ = {'I1_0': bool(I1_0), 'I1_1': bool(I1_1), 'I1_2': bool(I1_2), 'I1_3': bool(I1_3), 'adopt': ADOPT,
        'trunc': {'f1_style': TRUNC_F1, 'i1': TRUNC_I1}, 'p95_ms': P95, 'drive_weights_ok': DRIVE_OK,
        'calL_compare': CMP}
json.dump(summ, open(os.path.join(LOCAL, 'i1_summary.json'), 'w'), default=float, ensure_ascii=False, indent=1)
cmp_tab.to_csv(os.path.join(LOCAL, 'i1_calL.csv'), index=False)
dev_tab.to_csv(os.path.join(LOCAL, 'i1_devL.csv'), index=False)
if DRIVE_OK: to_drive()
for f in ['i1_summary.json', 'i1_calL.csv', 'i1_devL.csv']:
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(I1_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/i1_judge', 'zip', LOCAL)
print(f'zip {os.path.getsize(zp) / 2**20:.0f}MB:', zp)
from google.colab import files
files.download(zp)
print(json.dumps({k: summ[k] for k in ['I1_0', 'I1_1', 'I1_2', 'I1_3', 'adopt', 'trunc', 'p95_ms']}, ensure_ascii=False, indent=1, default=float))